# 🔥 Phase 2: Fine-tune Phi-3-mini with QLoRA
### PyTorch Debug Assistant

**What this notebook does:**
1. Installs dependencies
2. Loads `zehansunesara/pytorch-debug-assistant` from HuggingFace
3. Loads Phi-3-mini-4k-instruct in 4-bit (QLoRA)
4. Fine-tunes with `SFTTrainer`
5. Pushes the adapter to HuggingFace Hub

**Runtime:** Set to GPU T4 x2 in Kaggle Settings

## Step 1 — Install Dependencies

In [1]:
%%capture
!pip install -q -U huggingface_hub
!pip install -q -U bitsandbytes
!pip install -q triton
!pip install -q transformers==4.44.0
!pip install -q peft==0.12.0
!pip install -q trl==0.10.1
!pip install -q datasets==2.21.0
!pip install -q accelerate==0.34.2
!pip install -q wandb

## Step 2 — Authenticate with HuggingFace

In [ ]:
import os

def get_secret(name):
    # Try Kaggle first
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    # Try Colab
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name, "")

HF_TOKEN    = get_secret("HF_TOKEN")
WANDB_TOKEN = get_secret("WANDB_API_KEY")

from huggingface_hub import login
import wandb

login(token=HF_TOKEN)
print("HuggingFace login ✅")

wandb.login(key=WANDB_TOKEN)
print("W&B login ✅")

## Step 3 — Check GPU

In [3]:
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'CUDA: {torch.version.cuda}')

GPU: Tesla T4
VRAM: 15.6 GB
CUDA: 12.8


## Step 4 — Load and Inspect Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset('zehansunesara/pytorch-debug-assistant')
print(dataset)
print('\n--- Sample ---')
print('INPUT:', dataset['train'][0]['input'][:300])
print('\nOUTPUT:', dataset['train'][0]['output'][:300])

## Step 5 — Format into Chat Template

Phi-3-mini uses `<|user|>...<|end|>` format. We convert our instruction/input/output triplets into this format.

In [ ]:
def format_prompt(example):
    """
    Phi-3-mini instruction format:
    <|user|>\n{instruction}\n\n{input}<|end|>\n<|assistant|>\n{output}<|end|>
    """
    text = (
        f"<|user|>\n{example['instruction']}\n\n{example['input']}<|end|>\n"
        f"<|assistant|>\n{example['output']}<|end|>"
    )
    return {'text': text}

# Apply formatting
train_dataset = dataset['train'].map(format_prompt)
val_dataset   = dataset['validation'].map(format_prompt)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')
print('\n--- Formatted sample ---')
print(train_dataset[0]['text'][:500])

## Step 6 — Load Phi-3-mini-4k-instruct in 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NormalFloat4 — best quality at 4-bit
    bnb_4bit_compute_dtype=torch.float16,  # T4 uses fp16
    bnb_4bit_use_double_quant=False,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # required for SFTTrainer

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False  # disable KV cache during training

print(f'Model loaded ✅')
print(f'Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Step 7 — Configure LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training (freezes base weights, casts norms to fp32)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                    # rank — higher = more capacity, more params
    lora_alpha=32,           # scaling factor (alpha/r = effective LR scale)
    target_modules=[         # which layers to add LoRA to
        'q_proj', 'k_proj', 'v_proj', 'o_proj',  # attention
        'gate_proj', 'up_proj', 'down_proj'       # MLP
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# Show trainable params — should be ~1-2% of total
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} ({100 * trainable / total:.2f}% of total)')

## Step 8 — Training Configuration

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./checkpoints',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,          # reduce from 3 to 2 to save time
    max_steps=200,
    per_device_eval_batch_size=2,
    gradient_checkpointing=True,      # trades compute for memory
    optim='paged_adamw_32bit',        # paged optimizer for QLoRA
    learning_rate=1e-4, 
    lr_scheduler_type='cosine',
    warmup_steps=50,
    warmup_steps=50,
    weight_decay=0.001,
    fp16=True,
    bf16=False,                        
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,               # keep only last 2 checkpoints
    load_best_model_at_end=True,
    report_to='wandb',                 # set to 'wandb' if you want tracking
    run_name='phi3-qlora-v2-200steps',
    seed=42,
)

## Step 9 — Train!

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    dataset_text_field='text',
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
)

print('Starting training...')
trainer.train()
print('Training complete ✅')

## Step 10 — Evaluate: Before vs After

In [ ]:
from transformers import pipeline

# Test prompt — a real PyTorch error
test_input = """RuntimeError: Expected all tensors to be on the same device, 
but found at least two devices, cuda:0 and cpu!

I'm getting this error when trying to compute loss in my training loop.
My model is on GPU but I think my labels might be on CPU."""

prompt = (
    f"<|user|>\nYou are a PyTorch debugging assistant. "
    f"Given an error message or problem description, explain the root cause and provide a fix.\n\n"
    f"{test_input}<|end|>\n<|assistant|>\n"
)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
)

output = pipe(prompt)[0]['generated_text']
# Extract only the assistant response
response = output.split('<|assistant|>')[-1].strip()
print('=== Fine-tuned Model Response ===')
print(response)

## Step 11 — Push Adapter to HuggingFace Hub

In [ ]:
HF_USERNAME = 'zehansunesara'  # your HF username
ADAPTER_REPO = 'pytorch-debug-assistant-phi3-mini'

# Save and push the LoRA adapter (not the full model — adapter is only ~100MB)
model.push_to_hub(f'{HF_USERNAME}/{ADAPTER_REPO}', token=HF_TOKEN)
tokenizer.push_to_hub(f'{HF_USERNAME}/{ADAPTER_REPO}', token=HF_TOKEN)

print(f'\n✅ Adapter pushed to huggingface.co/{HF_USERNAME}/{ADAPTER_REPO}')
print('Note: This is just the LoRA adapter (~100MB), not the full 7B model.')
print('Users load it on top of microsoft/Phi-3-mini-4k-instruct')

## Step 12 — Save Training Loss Plot

In [ ]:
import matplotlib.pyplot as plt

# Extract loss history from trainer logs
logs = trainer.state.log_history
train_loss = [(l['step'], l['loss']) for l in logs if 'loss' in l]
eval_loss  = [(l['step'], l['eval_loss']) for l in logs if 'eval_loss' in l]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(*zip(*train_loss), label='Train Loss', color='steelblue')
ax.plot(*zip(*eval_loss),  label='Val Loss',   color='coral', linestyle='--')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('QLoRA Fine-tuning: Phi-3-mini on PyTorch Debug Assistant')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150)
plt.show()
print('Saved training_loss.png — add this to your README!')

## ✅ Phase 2 Complete!

**What you've done:**
- Fine-tuned Phi-3-mini with QLoRA on 809 high-quality PyTorch Q&A pairs
- Only ~1-2% of parameters were trained (LoRA adapters)
- Adapter pushed to HuggingFace Hub

**Next steps (Phase 3):**
- Apply GPTQ/AWQ post-training quantization
- Benchmark latency vs quality at different bit widths
- Build FastAPI + Gradio serving layer